In [1]:
import pathlib
import sys

_root = pathlib.Path().resolve()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
if str(_root.parent) not in sys.path and (_root.parent / 'src').exists():
    sys.path.insert(0, str(_root.parent))


# 00 Data Quality Assessment

**Sources:** `raw.*` (DuckDB), regenerated `outputs/findings/data_quality.json`. **Grain:** checks at source-table grain.

Notes: `raw.reviews.review_id` is not unique as a standalone key (duplicate IDs linked to different orders observed). Analytics preserves review-order pairs with grain `(review_id, order_id)` and deterministic handling of exact review-order rows.

In [2]:
import json

from src.db import get_connection
from src.validate import run_all_checks, save_findings

conn = get_connection(read_only=True)
try:
    results = run_all_checks(conn)
    out = save_findings(results)
    print('Saved:', out)
    print(json.dumps(results['row_counts'], indent=2))
finally:
    conn.close()

Saved: D:\Projects\olist-consumer-intelligence\outputs\findings\data_quality.json
{
  "orders": 99441,
  "order_items": 112650,
  "customers": 99441,
  "products": 32951,
  "sellers": 3095,
  "payments": 103886,
  "reviews": 99224,
  "geolocation": 1000163,
  "category_translation": 71
}


In [3]:
print('review_score_range:', results['review_score_range'])
print('primary_key_uniqueness:', results['primary_key_uniqueness'])
print('products.category null rate:', results['null_rates']['products']['product_category_name'])
print('referential_integrity sample:', {k: v for k, v in list(results['referential_integrity'].items())[:3]})

review_score_range: {'total': 99224, 'null': 0, 'non_numeric': 0, 'out_of_range': 0, 'invalid': 0, 'valid': 99224, 'valid_rate': 1.0}
primary_key_uniqueness: {'orders': True, 'order_items': True, 'customers': True, 'products': True, 'sellers': True, 'reviews': False}
products.category null rate: 0.018512336499651005
referential_integrity sample: {'order_items.order_id -> orders.order_id': {'total': 112650, 'null_fk': 0, 'orphaned': 0, 'integrity_rate': 1.0}, 'order_items.product_id -> products.product_id': {'total': 112650, 'null_fk': 0, 'orphaned': 0, 'integrity_rate': 1.0}, 'order_items.seller_id -> sellers.seller_id': {'total': 112650, 'null_fk': 0, 'orphaned': 0, 'integrity_rate': 1.0}}


Interpretation: referential integrity is assessed over non-null foreign keys with robust anti-join logic; review score validation reports total/null/non-numeric/out-of-range/valid counts. Grain and transform reconciliations are validated in `docs/olist/analytics_validation.json` and pipeline grain checks.